# Assignment: Pandas Case Study

## Part 1: Indian Startup Funding Data Analysis

---

### Introduction

In this case study, we will analyze **Indian Startup Funding Data** to extract meaningful insights about the startup ecosystem in India. This dataset contains information about various startups, their funding rounds, investors, and the amount of funding received.

**What you'll learn:**
- Loading and inspecting real-world CSV data
- Data cleaning techniques (handling NaN, removing commas, converting data types)
- Currency conversion (USD to INR)
- Datetime handling and component extraction
- GroupBy operations for aggregation
- Filtering data with boolean indexing
- Visualization with Matplotlib

### Dataset Columns

| Column | Description |
|--------|-------------|
| `SNo` | Serial number |
| `Date` | Date of funding round |
| `StartupName` | Name of the startup |
| `IndustryVertical` | Industry of the startup |
| `SubVertical` | Sub-category within the industry |
| `CityLocation` | City where the startup is based |
| `InvestorsName` | Name of the investors |
| `InvestmentType` | Type of investment (Private Equity, Seed Funding, etc.) |
| `AmountInUSD` | Funding amount in US Dollars |
| `Remarks` | Additional remarks |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

---

## Loading the Data

We load the dataset directly from a GitHub raw URL. The `read_csv()` function reads comma-separated value files into a Pandas DataFrame.

> **Note:** We use `encoding='utf-8'` to handle special characters in the dataset.

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/Pranav892/Datasets/main/startup_funding.csv', encoding='utf-8')
df

---

## Understanding the Dataset

Before doing any analysis, always understand your data first using:
- `.shape` - Number of rows and columns
- `.info()` - Data types and non-null counts
- `.describe()` - Statistical summary of numeric columns
- `.head()` - First 5 rows

In [ ]:
df.shape

In [ ]:
df.info()

---

## Data Cleaning - Renaming Columns

Real-world datasets often have inconsistent column names. We rename them to a **snake_case** convention for consistency and readability.

The `rename()` method takes a dictionary mapping old names to new names. We use `inplace=True` to modify the DataFrame directly.

In [ ]:
df.rename(columns={
    'SNo': 'sno',
    'Date': 'date',
    'StartupName': 'startup_name',
    'IndustryVertical': 'industry_vertical',
    'SubVertical': 'sub_vertical',
    'CityLocation': 'city_location',
    'InvestorsName': 'investors_name',
    'InvestmentType': 'investment_type',
    'AmountInUSD': 'amount_in_usd',
    'Remarks': 'remarks'
}, inplace=True)

df.head()

---

## Cleaning the Amount Column

The `amount_in_usd` column has several issues:
1. **Commas** in numbers (e.g., `1,000,000`)
2. **`undisclosed`** values instead of numbers
3. Values are **strings** instead of numbers

We need to clean all of these before we can do any numerical analysis.

In [ ]:
df['amount_in_usd'].apply(lambda x: float(str(x).replace(',', '')))

### Handling Non-Numeric Values

We write a function that:
1. Converts the value to string (in case it's already a number or NaN)
2. Removes commas
3. Replaces 'undisclosed' with 'nan' (Not a Number)
4. Converts to float

In [ ]:
def remove_comma(x):
    return float(str(x).replace(',', '').replace('undisclosed', 'nan'))

df['amount_in_usd'] = df['amount_in_usd'].apply(remove_comma)
df['amount_in_usd']

---

## Converting USD to INR

Since this is Indian startup data, let's convert the funding amounts from **USD to INR** for better context.

We use the `apply()` method with a custom function to perform the conversion row by row.

In [ ]:
def to_inr(dollar):
    usd_to_inr = 81.50  # approximate exchange rate
    return dollar * usd_to_inr

df['amount_in_usd'].apply(to_inr)

In [ ]:
df['amount_in_inr'] = df['amount_in_usd'].apply(to_inr)
df

---

## Converting Date Column

The `date` column is currently a string. We convert it to **datetime** format using `pd.to_datetime()`, which enables powerful time-based operations:
- Extract year, month, day
- Filter by date ranges
- Group by time periods

In [ ]:
df['date'] = pd.to_datetime(df['date'])
df.dtypes

In [ ]:
df['date'].dt.year
df['date'].dt.month
df['date'].dt.day

---

## Analysis: IDG Ventures Investment

Let's analyze the investments made by **IDG Ventures India**, a prominent venture capital firm.

We use **boolean indexing** to filter rows where the investor is 'IDG Ventures India', then sum up the total investment amount.

In [ ]:
df[df['investors_name'] == 'IDG Ventures India']
df[df['investors_name'] == 'IDG Ventures India']['amount_in_inr'].sum()

In [ ]:
idg_df = df[df['investors_name'] == 'IDG Ventures India']
print("Total Investment by IDG Ventures India:", idg_df['amount_in_inr'].sum(), 'INR')
print("Number of Investments:", len(idg_df))

---

## Yearly Investment Trend

Using `groupby()` on the date, we can see how total investments have changed over the years. This helps identify **trends** in the startup ecosystem.

In [ ]:
df.groupby('date')['amount_in_inr'].sum()

In [ ]:
plt.figure(figsize=(10, 6))
df.groupby(df['date'].dt.year)['amount_in_inr'].sum().plot(kind='bar')
plt.xlabel('Year')
plt.ylabel('Investment Amount (INR)')
plt.title('Yearly Investment Trend')
plt.show()

---

## Top Funded Startups

Let's find the **top 10 most funded startups** using `groupby()` on startup name, summing the amounts, and sorting in descending order.

In [ ]:
top_startups = df.groupby('startup_name')['amount_in_inr'].sum().sort_values(ascending=False).head(10)
top_startups

In [ ]:
plt.figure(figsize=(12, 6))
top_startups.plot(kind='bar')
plt.xlabel('Startup Name')
plt.ylabel('Total Investment (INR)')
plt.title('Top 10 Most Funded Startups')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

---

## Monthly Investment Trend

We extract the **month** from the datetime column and aggregate total investments to see which months see the most funding activity.

In [ ]:
df.groupby(df['date'].dt.month)['amount_in_inr'].sum()

In [ ]:
plt.figure(figsize=(10, 6))
df.groupby(df['date'].dt.month)['amount_in_inr'].sum().plot(kind='bar')
plt.xlabel('Month')
plt.ylabel('Investment Amount (INR)')
plt.title('Monthly Investment Trend')
plt.xticks(rotation=0)
plt.show()

---

## Industry Analysis

Using `value_counts()` we can see which industries receive the most funding and which have the most startups.

In [ ]:
df['industry_vertical'].value_counts()

In [ ]:
df.groupby('industry_vertical')['amount_in_inr'].sum().sort_values(ascending=False).head(10)

---

## City Analysis

Geographic distribution of startups shows which cities are the **startup hubs** of India.

In [ ]:
df['city_location'].value_counts().head(10)

---

## Investment Type Analysis

Different types of investments (Seed Funding, Private Equity, Angel Funding, etc.) represent different stages of a startup's lifecycle.

In [ ]:
df['investment_type'].value_counts()

In [ ]:
df.groupby('investment_type')['amount_in_inr'].sum().sort_values(ascending=False)

---

## Key Takeaways - Part 1

1. **Data cleaning is crucial** - Real-world data has commas, 'undisclosed' values, and mixed types
2. **Datetime conversion** enables powerful time-based analysis (yearly, monthly trends)
3. **Currency conversion** gives context to international datasets
4. **GroupBy operations** are the backbone of aggregation analysis
5. **Visualization** helps identify trends that numbers alone can't show
6. **Boolean indexing** is essential for filtering specific subsets of data

---

# Part 2: Pandas Case Study - NLP Data Analysis

## IMDB Movie Reviews Dataset

---

### Introduction

In this case study, we'll use Pandas to perform **text data preprocessing** for Natural Language Processing (NLP). The IMDB dataset contains 50,000 movie reviews labeled as positive or negative.

**What you'll learn:**
- Text cleaning techniques (lowercase, strip, remove HTML/URLs)
- Handling contractions/abbreviations
- Tokenization using NLTK
- Stopwords removal
- Text length analysis
- WordCloud visualization
- Bag of Words model using CountVectorizer
- PCA for dimensionality reduction and visualization

### Dataset

| Column | Description |
|--------|-------------|
| `review` | The full text of the movie review |
| `sentiment` | Label: `positive` or `negative` |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/Pranav892/Datasets/main/imdb_dataset.csv')
df.head()

In [ ]:
df.shape
df.info()

---

## Text Cleaning Step 1: Lowercase Conversion

Converting all text to **lowercase** ensures that words like "Good", "good", and "GOOD" are treated as the same word. This is a fundamental preprocessing step in NLP.

We use the Pandas `.str.lower()` accessor to apply string methods to an entire column.

In [ ]:
df['review'] = df['review'].str.lower()
df.head()

---

## Text Cleaning Step 2: Strip Whitespace

Leading and trailing whitespace can cause issues during tokenization and comparison. The `.str.strip()` method removes them.

In [ ]:
df['review'] = df['review'].str.strip()

---

## Text Cleaning Step 3: Remove HTML Tags

Since IMDB reviews are scraped from web pages, they often contain **HTML tags** like `<br />`, `<a>`, etc. These tags don't carry sentiment and must be removed.

We use **regular expressions** (regex) to match and remove all HTML tags.

The pattern `<.*?>` matches:
- `<` - opening bracket
- `.*?` - any characters (non-greedy)
- `>` - closing bracket

In [ ]:
import re

def remove_html_tags(text):
    pattern = re.compile('<.*?>')
    return pattern.sub(r'', text)

df['review'] = df['review'].apply(remove_html_tags)
df.head()

---

## Text Cleaning Step 4: Remove URLs

URLs in text don't contribute to sentiment analysis. We remove them using regex.

The pattern `https?://\S+|www\.\S+` matches:
- `https://` or `http://` followed by non-space characters
- `www.` followed by non-space characters

In [ ]:
def remove_url(text):
    pattern = re.compile(r'https?://\S+|www\.\S+')
    return pattern.sub(r'', text)

df['review'] = df['review'].apply(remove_url)

---

## Text Cleaning Step 5: Expand Abbreviations (Contractions)

English text often uses **contractions** like:
- `don't` -> `do not`
- `can't` -> `cannot`
- `I'm` -> `I am`

We use the `contractions` library to expand these into their full forms. This helps because "don't" and "do not" would otherwise be treated as different tokens.

In [ ]:
import contractions

def expand_contractions(text):
    return contractions.fix(text)

df['review'] = df['review'].apply(expand_contractions)

---

## Text Cleaning Step 6: Remove Punctuation

Punctuation marks (`.`, `,`, `!`, `?`, etc.) generally don't add meaning for sentiment analysis. We remove them using Python's `string.punctuation` constant.

`str.maketrans()` creates a translation table, and `translate()` applies it to remove all punctuation characters.

In [ ]:
import string

def remove_punctuation(text):
    translator = str.maketrans('', '', string.punctuation)
    return text.translate(translator)

df['review'] = df['review'].apply(remove_punctuation)

---

## Tokenization

**Tokenization** is the process of splitting text into individual words (tokens). This is a fundamental step in NLP.

For example: `"this movie is great"` -> `['this', 'movie', 'is', 'great']`

We use NLTK's `word_tokenize()` for this.

In [ ]:
from nltk.tokenize import word_tokenize

df['tokens'] = df['review'].apply(word_tokenize)
df.head()

---

## Stopwords Removal

**Stopwords** are common words that appear frequently but carry little meaning:
- Articles: `the`, `a`, `an`
- Prepositions: `in`, `on`, `at`, `by`
- Pronouns: `he`, `she`, `it`
- Conjunctions: `and`, `or`, `but`

Removing these helps focus on the **meaningful words** that determine sentiment.

In [ ]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def remove_stopwords(tokens):
    return [word for word in tokens if word not in stop_words]

df['tokens'] = df['tokens'].apply(remove_stopwords)
df.head()

---

## Character and Word Length Analysis

Analyzing the **length of reviews** can give insights:
- Do positive reviews tend to be longer than negative ones?
- What's the typical review length?

We use `.str.len()` for character count and `.str.split().str.len()` for word count.

In [ ]:
df['char_length'] = df['review'].str.len()
df.head()

In [ ]:
df['word_length'] = df['review'].str.split().str.len()
df.head()

---

## Distribution Plots

Histograms with KDE (Kernel Density Estimation) help us understand the **distribution** of review lengths. This can reveal patterns like whether reviews are typically short or long.

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df['word_length'], bins=50, kde=True)
plt.xlabel('Word Count')
plt.ylabel('Frequency')
plt.title('Distribution of Word Count in Reviews')
plt.show()

---

## WordCloud Visualization

A **WordCloud** is a visual representation of text data where the size of each word indicates its frequency. Larger words appear more frequently in the corpus.

This helps quickly identify the **most common words** in positive vs negative reviews.

In [ ]:
from wordcloud import WordCloud

def generate_wordcloud(text):
    wordcloud = WordCloud(width=800, height=400, background_color='white').generate(text)
    plt.figure(figsize=(10, 6))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.show()

In [ ]:
positive_text = ' '.join(df[df['sentiment'] == 'positive']['review'].values)
generate_wordcloud(positive_text)

In [ ]:
negative_text = ' '.join(df[df['sentiment'] == 'negative']['review'].values)
generate_wordcloud(negative_text)

---

## N-Grams Analysis

**N-grams** are contiguous sequences of N words in text:
- **Unigrams (1-gram):** individual words
- **Bigrams (2-grams):** pairs of words (e.g., "very good", "not bad")
- **Trigrams (3-grams):** triplets of words

N-grams capture **word context** that single words miss. For example, "not good" has a very different meaning than "good" alone.

We use `CountVectorizer` with `ngram_range` parameter to extract n-grams.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(ngram_range=(2, 2))
ngram_matrix = cv.fit_transform(df['review'])
ngram_matrix

### Bigram Frequency

Let's find the most common bigrams (2-word combinations) in the dataset.

In [ ]:
bigram_freq = ngram_matrix.sum(axis=0).A1
bigram_names = cv.get_feature_names_out()
bigram_df = pd.DataFrame({'bigram': bigram_names, 'frequency': bigram_freq})
bigram_df.sort_values('frequency', ascending=False).head(20)

---

## Bag of Words (BoW) - CountVectorizer

**Bag of Words** is one of the simplest text vectorization techniques. It creates a matrix where:
- Each **row** represents a document (review)
- Each **column** represents a unique word
- Each **cell** contains the count of that word in that document

> The name "Bag of Words" comes from the idea that we treat the document as a "bag" of words, ignoring word order.

### How CountVectorizer Works:
1. Build a vocabulary of all unique words
2. For each document, count the occurrence of each word
3. Create a matrix of document-word counts

In [ ]:
cv = CountVectorizer()
bow_matrix = cv.fit_transform(df['review'])
bow_matrix

In [ ]:
bow_df = pd.DataFrame(bow_matrix.toarray(), columns=cv.get_feature_names_out())
bow_df.head()

### Vocabulary Size

In [ ]:
print("Vocabulary size:", len(cv.vocabulary_))

---

## PCA Visualization

**PCA (Principal Component Analysis)** reduces high-dimensional data (like the BoW matrix with thousands of columns) to 2 dimensions for visualization.

This helps us see if positive and negative reviews form **distinct clusters** in the vector space.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
bow_pca = pca.fit_transform(bow_matrix.toarray())

pca_df = pd.DataFrame({
    'PC1': bow_pca[:, 0],
    'PC2': bow_pca[:, 1],
    'sentiment': df['sentiment']
})
pca_df.head()

In [ ]:
plt.figure(figsize=(10, 8))
sns.scatterplot(data=pca_df, x='PC1', y='PC2', hue='sentiment', alpha=0.5)
plt.title('PCA Visualization of IMDB Reviews (Bag of Words)')
plt.show()

---

## Key Takeaways - Part 2

1. **Text cleaning is multi-step:** lowercase -> strip -> remove HTML/URLs -> expand contractions -> remove punctuation -> tokenize -> remove stopwords
2. **Pandas `.str` accessor** makes text operations on entire columns efficient and concise
3. **`.apply()` with custom functions** is the key method for text preprocessing in Pandas
4. **WordCloud** provides quick visual insights into word frequency
5. **N-grams** capture word context ("not good" vs "good") that unigrams miss
6. **Bag of Words** is simple but effective for text classification
7. **PCA** helps visualize high-dimensional text data in 2D
8. **Text data requires careful preprocessing** before any NLP model can be applied

---

## Summary of All Pandas Functions Used

| Function | Purpose |
|----------|--------|
| `pd.read_csv()` | Load CSV data |
| `.shape` | Get dimensions |
| `.info()` | Data types and null counts |
| `.rename()` | Rename columns |
| `.apply()` | Apply function to each element |
| `.groupby()` | Group data for aggregation |
| `.sort_values()` | Sort by values |
| `.value_counts()` | Count unique values |
| `.str.lower()` | Convert to lowercase |
| `.str.strip()` | Remove whitespace |
| `.str.len()` | String length |
| `.str.split()` | Split strings |
| `pd.to_datetime()` | Convert to datetime |
| `.dt.year/.month/.day` | Extract date components |